# Machine Learning in Solar Physics — Implementation / 구현

**Paper**: Asensio Ramos, A., Cheung, M. C. M., Chifu, I., & Gafeira, R. (2023). *Living Reviews in Solar Physics*, 20:4. DOI: 10.1007/s41116-023-00038-x

**English.** This notebook reproduces the key algorithmic ideas surveyed in the paper using lightweight NumPy / scikit-learn implementations, so that every step is inspectable and no heavy deep-learning dependency is required. It covers six vignettes:

1. MLP for Stokes → atmospheric-parameter retrieval (toy Milne-Eddington)
2. U-Net schematic for solar segmentation
3. CNN feature-extraction visualization (learned vs. handcrafted filters)
4. Flare prediction: logistic regression baseline vs. a small CNN proxy on SHARP-like features
5. Super-resolution demo: bicubic vs. CNN (simple residual CNN)
6. Physics-informed neural network (PINN) minimal example: ODE residual + data loss

**한국어.** 본 노트북은 논문에서 다룬 주요 알고리즘 아이디어들을 NumPy/scikit-learn으로 가볍게 구현하여 각 단계가 투명하게 확인 가능하도록 구성한다. 무거운 DL 프레임워크 의존성 없이 Stokes 인버전·분할·CNN 필터·플레어 예측·초해상도·PINN의 6가지 시나리오를 다룬다.

In [ ]:
"""Imports and global plotting configuration.

Uses only NumPy, Matplotlib, and scikit-learn to keep the notebook
executable in any Python environment with the `study-with-ai` conda env.
"""
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
RNG = np.random.default_rng(42)

## Part 1 / 파트 1: MLP for Stokes → Atmospheric-Parameter Retrieval

**English.** The paper's workhorse example is Stokes-profile inversion: given $(I, Q, U, V)(\lambda)$, infer atmospheric parameters $(B, \theta, \phi, v, T)$. Classical codes (SIR, SNAPI) iterate per pixel. We demonstrate the ML shortcut with a toy Milne-Eddington forward model and a small MLP inverse model. SICON (Asensio Ramos & Díaz Baso 2019) delivers a ~$10^5$× speedup on 512×512 Hinode maps (200 ms/map); our tiny experiment shows the same paradigm at a much smaller scale.

**한국어.** 논문의 대표 예시는 Stokes 인버전: 관측된 $(I, Q, U, V)(\lambda)$로부터 대기 변수 $(B, \theta, \phi, v, T)$를 추론. 전통적 코드(SIR, SNAPI)는 픽셀별 반복 최적화. 여기서는 Milne-Eddington 단순 forward 모델과 작은 MLP 역방향 모델로 같은 패러다임을 시연.

In [ ]:
def milne_eddington_forward(params: np.ndarray, wavelengths: np.ndarray) -> np.ndarray:
    """Toy Milne-Eddington forward model producing Stokes I and V.

    Args:
        params: Array of shape (N, 3) with columns [B_field, v_doppler, eta_0].
        wavelengths: 1D array of wavelength offsets from line center (in units of
            Doppler width).

    Returns:
        Array of shape (N, 2 * len(wavelengths)) containing concatenated
        Stokes I and V profiles (I first, then V).
    """
    B, v, eta0 = params[:, 0:1], params[:, 1:2], params[:, 2:3]
    wl = wavelengths[np.newaxis, :]
    # Absorption (Gaussian) profile H shifted by v.
    H = np.exp(-((wl - v) ** 2))
    # Stokes I = 1 - eta0 * H.
    I = 1.0 - eta0 * H
    # Zeeman V ~ -B * dH/dl (weak-field proxy).
    dH_dl = -2.0 * (wl - v) * H
    V = -B * eta0 * dH_dl * 0.1
    return np.hstack([I, V])


# Generate training data: sample random atmospheric parameters.
N_SAMPLES = 4000
wl_grid = np.linspace(-3.0, 3.0, 31)
true_params = np.column_stack([
    RNG.uniform(0.0, 3.0, N_SAMPLES),   # B: 0-3 kG (normalized)
    RNG.uniform(-1.5, 1.5, N_SAMPLES),  # Doppler shift
    RNG.uniform(0.3, 2.0, N_SAMPLES),   # line opacity eta0
])
stokes = milne_eddington_forward(true_params, wl_grid)
# Add realistic noise (1% of continuum).
stokes_noisy = stokes + 0.01 * RNG.standard_normal(stokes.shape)
print(f'Stokes feature shape: {stokes_noisy.shape}')
print(f'Target parameter shape: {true_params.shape}')

In [ ]:
"""Train an MLP as the inverse Stokes solver.

The MLP has two hidden layers of 64 neurons each with ReLU activations,
replicating the style of Carroll & Staude (2001) but with modern defaults.
"""
X_tr, X_te, y_tr, y_te = train_test_split(stokes_noisy, true_params, test_size=0.25, random_state=0)
scaler_X = StandardScaler().fit(X_tr)
scaler_y = StandardScaler().fit(y_tr)

mlp = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    solver='adam',
    max_iter=200,
    random_state=0,
    early_stopping=True,
)
mlp.fit(scaler_X.transform(X_tr), scaler_y.transform(y_tr))
y_pred = scaler_y.inverse_transform(mlp.predict(scaler_X.transform(X_te)))

# Per-parameter RMSE.
labels = ['B (kG)', 'v_Doppler', 'eta0']
for i, lab in enumerate(labels):
    rmse = float(np.sqrt(np.mean((y_pred[:, i] - y_te[:, i]) ** 2)))
    print(f'RMSE {lab:>10}: {rmse:.4f}')

In [ ]:
"""Visualize predicted vs. true atmospheric parameters."""
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (ax, lab) in enumerate(zip(axes, labels)):
    ax.scatter(y_te[:, i], y_pred[:, i], s=4, alpha=0.3)
    lo, hi = y_te[:, i].min(), y_te[:, i].max()
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_xlabel(f'True {lab}')
    ax.set_ylabel(f'Predicted {lab}')
    ax.set_title(f'MLP inversion: {lab}')
fig.suptitle('Part 1: MLP Stokes inversion (toy ME model) / MLP 스토크스 인버전')
plt.tight_layout()
plt.show()

## Part 2 / 파트 2: U-Net Schematic for Solar Segmentation

**English.** U-Net (Ronneberger et al. 2015) is the workhorse segmentation architecture used by Illarionov & Tlatov (2018) for coronal-hole detection on AIA 193 Å. We don't train a full U-Net here but produce an **ASCII + matplotlib schematic** showing channel counts (C → 2C → 4C → 8C → 16C) and skip connections, matching Fig 7 of the paper.

**한국어.** U-Net은 Ronneberger 등(2015)의 구조로, Illarionov & Tlatov(2018)이 AIA 193 Å 코로나홀 검출에 사용한 분할의 표준. 여기서는 전체 학습 대신 채널 수(C → 16C)와 skip-connection 구조를 모식도로 시각화.

In [ ]:
"""Draw a U-Net schematic matching the paper's Fig 7."""
fig, ax = plt.subplots(figsize=(12, 6))

# Encoder path (left): rectangles with decreasing height, increasing channel count.
levels = [('C', 3.0, 0), ('2C', 2.0, 1), ('4C', 1.3, 2), ('8C', 0.8, 3), ('16C', 0.5, 4)]
enc_y = 5.0
for lbl, h, i in levels:
    ax.add_patch(plt.Rectangle((i * 1.5, enc_y - h / 2), 0.8, h,
                                facecolor='#a8d8ea', edgecolor='k'))
    ax.text(i * 1.5 + 0.4, enc_y, lbl, ha='center', va='center', fontsize=10)

# Decoder path (right): mirror of encoder.
for lbl, h, i in levels[:-1]:
    x = 8 + i * 1.5
    ax.add_patch(plt.Rectangle((x, enc_y - h / 2), 0.8, h,
                                facecolor='#aa96da', edgecolor='k'))
    ax.text(x + 0.4, enc_y, lbl, ha='center', va='center', fontsize=10)

# Skip connections.
for i, (lbl, h, _) in enumerate(levels[:-1]):
    x_enc = i * 1.5 + 0.8
    x_dec = 8 + (len(levels) - 2 - i) * 1.5
    ax.annotate('', xy=(x_dec, enc_y + 0.05), xytext=(x_enc, enc_y + 0.05),
                arrowprops=dict(arrowstyle='->', color='gray', ls='--'))

# Input and output annotations.
ax.text(0.4, 6.8, 'Input\n(3ch)', ha='center', fontsize=10, color='black')
ax.text(8 + 3 * 1.5 + 0.4, 6.8, 'Output\n(1ch mask)', ha='center', fontsize=10, color='black')
ax.text(4 * 1.5 + 0.4, 2.8, 'Bottleneck', ha='center', fontsize=10, color='black', style='italic')

ax.set_xlim(-0.5, 14)
ax.set_ylim(1.5, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Part 2: U-Net schematic (blue=encoder, purple=decoder, dashed=skip)\n'
             'U-Net 모식도 (파랑=인코더, 보라=디코더, 점선=스킵 연결)')
plt.tight_layout()
plt.show()

## Part 3 / 파트 3: CNN Feature-Extraction Visualization (Filters)

**English.** CNNs learn convolution kernels that extract features such as edges, blobs, and oriented gradients. Following the paper's discussion of Huang et al. (2018) — whose flare-prediction CNNs developed PIL-sensitive intermediate filters — we visualize both handcrafted kernels (edge, Laplacian, Gaussian blur) and a set of random 3×3 kernels applied to a synthetic solar granulation image.

**한국어.** CNN은 에지, blob, 방향성 gradient 같은 특징을 추출하는 커널을 학습. 논문 §7.3.6의 Huang 등(2018)은 플레어 예측 CNN이 PIL-민감 공간 필터를 학습함을 관측. 합성 granulation 영상에 수작업 커널과 무작위 3×3 커널을 적용해 필터 개념을 시각화.

In [ ]:
def convolve2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Simple 2D convolution with zero padding.

    Args:
        image: 2D input array.
        kernel: 2D kernel array (must be odd-sized).

    Returns:
        2D output array of the same shape as the input.
    """
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='edge')
    out = np.zeros_like(image)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            out[i, j] = float(np.sum(padded[i:i + kh, j:j + kw] * kernel))
    return out


# Synthetic solar granulation pattern.
N = 64
xx, yy = np.meshgrid(np.linspace(-3, 3, N), np.linspace(-3, 3, N))
granulation = 0.0
for _ in range(18):
    cx, cy = RNG.uniform(-3, 3, 2)
    granulation = granulation + np.exp(-2.0 * ((xx - cx) ** 2 + (yy - cy) ** 2))
granulation = granulation + 0.2 * RNG.standard_normal(granulation.shape)

# Handcrafted kernels.
edge_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
edge_y = edge_x.T
laplacian = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=float)
blur = np.ones((3, 3)) / 9.0

kernels = [('Input', None), ('Sobel-X', edge_x), ('Sobel-Y', edge_y),
           ('Laplacian', laplacian), ('Blur', blur)]
# Plus three random kernels.
for k in range(3):
    kernels.append((f'Random-{k + 1}', RNG.standard_normal((3, 3))))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (name, K) in zip(axes.flat, kernels):
    out = granulation if K is None else convolve2d(granulation, K)
    ax.imshow(out, cmap='gray')
    ax.set_title(name, fontsize=10)
    ax.axis('off')
fig.suptitle('Part 3: CNN filters on synthetic granulation / 합성 granulation에 적용한 CNN 필터')
plt.tight_layout()
plt.show()

## Part 4 / 파트 4: Flare Prediction — Logistic Regression vs. CNN Proxy on SHARP-like Features

**English.** Following Bobra & Couvidat (2015) and Liu et al. (2019), we simulate SHARP-like feature vectors (area, total unsigned flux, current helicity, twist, PIL length) for flaring and non-flaring active regions. We train two baselines:
- Logistic regression on raw SHARP features (classical ML baseline in the paper).
- An MLP (proxy for a CNN on magnetogram patches) with the same features.

We evaluate using **TSS (True Skill Statistic)** and **HSS** defined in §7.3.2 of the paper.

**한국어.** Bobra & Couvidat(2015), Liu 등(2019)을 따라 flare·non-flare AR의 SHARP 유사 특징(면적, 총 자속, current helicity, twist, PIL 길이)을 합성. 로지스틱 회귀(논문의 고전 베이스라인)와 MLP(자기도 패치용 CNN 프록시)를 비교. 논문 §7.3.2의 TSS, HSS로 평가.

In [ ]:
def skill_scores(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute binary flare-prediction skill scores.

    Matches definitions in Table 2 of Asensio Ramos et al. (2023).

    Args:
        y_true: Ground-truth binary labels (1=flare, 0=no flare).
        y_pred: Predicted binary labels.

    Returns:
        Dictionary with TP, FP, FN, TN, recall, far, tss, hss1 scalar entries.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    tss = recall - far
    hss1 = recall * (2.0 - 1.0 / precision) if precision > 0 else float('nan')
    return {'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
            'recall': recall, 'FAR': far, 'TSS': tss, 'HSS1': hss1}


# Simulate SHARP-like features for 2000 ARs, 15% flaring (realistic imbalance proxy).
N = 2000
frac_flare = 0.15
labels = (RNG.uniform(size=N) < frac_flare).astype(int)

# Non-flaring ARs: smaller area, lower twist/helicity. Flaring: higher.
area = np.where(labels == 1, RNG.gamma(4.0, 1.5, N), RNG.gamma(2.0, 1.0, N))
flux = np.where(labels == 1, RNG.gamma(3.5, 2.0, N), RNG.gamma(1.8, 1.5, N))
helicity = np.where(labels == 1, RNG.gamma(2.5, 1.0, N), RNG.gamma(1.2, 0.8, N))
twist = np.where(labels == 1, RNG.gamma(2.0, 0.6, N), RNG.gamma(1.0, 0.6, N))
pil = np.where(labels == 1, RNG.gamma(3.0, 2.5, N), RNG.gamma(1.5, 2.0, N))
features = np.column_stack([area, flux, helicity, twist, pil])
features = features + 0.3 * RNG.standard_normal(features.shape)  # measurement noise

X_tr, X_te, y_tr, y_te = train_test_split(features, labels, test_size=0.3,
                                           stratify=labels, random_state=0)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

# Baseline 1: logistic regression.
logreg = LogisticRegression(class_weight='balanced', max_iter=500).fit(X_tr_s, y_tr)
yp_log = logreg.predict(X_te_s)

# Baseline 2: MLP (proxy for a CNN on magnetogram patches).
mlp_clf = MLPClassifier(hidden_layer_sizes=(32, 32), max_iter=400, random_state=0,
                         early_stopping=True).fit(X_tr_s, y_tr)
yp_mlp = mlp_clf.predict(X_te_s)

print('Logistic regression:', skill_scores(y_te, yp_log))
print('MLP (CNN proxy)    :', skill_scores(y_te, yp_mlp))

In [ ]:
"""Visualize skill-score comparison of the two baselines."""
scores_log = skill_scores(y_te, yp_log)
scores_mlp = skill_scores(y_te, yp_mlp)
metrics = ['recall', 'FAR', 'TSS']
vals_log = [scores_log[m] for m in metrics]
vals_mlp = [scores_mlp[m] for m in metrics]

x = np.arange(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w / 2, vals_log, w, label='Logistic Regression', color='#4e79a7')
ax.bar(x + w / 2, vals_mlp, w, label='MLP (CNN proxy)', color='#f28e2b')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('Part 4: Flare prediction skill scores (TSS = Recall − FAR)\n'
             '플레어 예측 성능 지표 비교')
ax.axhline(0, color='k', lw=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## Part 5 / 파트 5: Super-Resolution — Bicubic vs. CNN (Residual Regression)

**English.** The paper's Enhance model (Díaz Baso & Asensio Ramos 2018) maps HMI 0.5″/pix → 0.25″/pix (×2 per axis, **×4 pixel count**) and approximately doubles continuum contrast vs. Hinode benchmarks. We build a miniature analogue: downsample a synthetic solar continuum image, then compare classical bicubic upsampling against a small MLP that maps low-resolution patches to high-resolution patches (a residual-CNN proxy).

**한국어.** 논문의 Enhance(Díaz Baso & Asensio Ramos 2018)는 HMI 0.5″/pix → 0.25″/pix(**×4 픽셀**)로 초해상화하며 Hinode 기준 대비 continuum contrast가 약 2배 개선. 여기서는 합성 태양 continuum 이미지를 다운샘플 후 (a) 고전 bicubic 업샘플링과 (b) low-resolution 패치를 high-resolution 패치로 매핑하는 소형 MLP(residual-CNN 프록시)를 비교.

In [ ]:
def bicubic_upsample_2x(low: np.ndarray) -> np.ndarray:
    """Basic 2x bicubic upsampling (scipy-free, using numpy polynomial interp).

    Args:
        low: 2D low-resolution image.

    Returns:
        2D array with 2x the side length of `low`.
    """
    H, W = low.shape
    # Upsample along rows.
    x_lo = np.arange(W)
    x_hi = np.linspace(0, W - 1, 2 * W)
    rows = np.array([np.interp(x_hi, x_lo, r) for r in low])
    # Upsample along columns.
    y_lo = np.arange(H)
    y_hi = np.linspace(0, H - 1, 2 * H)
    high = np.array([np.interp(y_hi, y_lo, c) for c in rows.T]).T
    return high


# Synthetic high-res continuum image (bright granules + darker lanes).
HR_SIZE = 64
xx, yy = np.meshgrid(np.linspace(-3, 3, HR_SIZE), np.linspace(-3, 3, HR_SIZE))
hr = np.zeros_like(xx)
for _ in range(30):
    cx, cy = RNG.uniform(-3, 3, 2)
    hr = hr + np.exp(-3.0 * ((xx - cx) ** 2 + (yy - cy) ** 2))
hr = (hr - hr.min()) / (hr.max() - hr.min())
# Downsample by 2x with simple averaging (models HMI 0.5″).
lr = 0.25 * (hr[::2, ::2] + hr[1::2, ::2] + hr[::2, 1::2] + hr[1::2, 1::2])
# Upsample baseline: bicubic.
hr_bicubic = bicubic_upsample_2x(lr)

print(f'HR shape: {hr.shape}; LR shape: {lr.shape}; bicubic SR shape: {hr_bicubic.shape}')

In [ ]:
"""Train a small MLP to map 3x3 LR patches to the corresponding 6x6 HR patches.

This is a simplified analogue of a residual CNN such as Enhance or SRCNN.
We generate many (LR-patch, HR-patch) pairs from a training set of synthetic
images, train the MLP, then apply it patch-by-patch to the test image.
"""
PATCH = 3
HR_PATCH = 2 * PATCH
N_TRAIN_IMAGES = 8
patches_lr = []
patches_hr = []
for _ in range(N_TRAIN_IMAGES):
    img = np.zeros_like(xx)
    for _ in range(30):
        cx, cy = RNG.uniform(-3, 3, 2)
        img = img + np.exp(-3.0 * ((xx - cx) ** 2 + (yy - cy) ** 2))
    img = (img - img.min()) / (img.max() - img.min())
    img_lr = 0.25 * (img[::2, ::2] + img[1::2, ::2] + img[::2, 1::2] + img[1::2, 1::2])
    # Sample random patches.
    for _ in range(200):
        i = int(RNG.integers(0, img_lr.shape[0] - PATCH))
        j = int(RNG.integers(0, img_lr.shape[1] - PATCH))
        lr_patch = img_lr[i:i + PATCH, j:j + PATCH]
        hr_patch = img[2 * i:2 * i + HR_PATCH, 2 * j:2 * j + HR_PATCH]
        patches_lr.append(lr_patch.ravel())
        patches_hr.append(hr_patch.ravel())

patches_lr = np.array(patches_lr)
patches_hr = np.array(patches_hr)

sr_mlp = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=200, random_state=0,
                      early_stopping=True).fit(patches_lr, patches_hr)

# Apply patch-wise to the test LR image.
hr_sr = np.zeros_like(hr)
counts = np.zeros_like(hr)
for i in range(lr.shape[0] - PATCH + 1):
    for j in range(lr.shape[1] - PATCH + 1):
        lr_patch = lr[i:i + PATCH, j:j + PATCH].ravel()
        pred = sr_mlp.predict(lr_patch.reshape(1, -1)).reshape(HR_PATCH, HR_PATCH)
        hr_sr[2 * i:2 * i + HR_PATCH, 2 * j:2 * j + HR_PATCH] += pred
        counts[2 * i:2 * i + HR_PATCH, 2 * j:2 * j + HR_PATCH] += 1
hr_sr = hr_sr / np.maximum(counts, 1)

# Compare contrast.
contrast_hr = float(np.std(hr))
contrast_bicubic = float(np.std(hr_bicubic))
contrast_mlp = float(np.std(hr_sr))
print(f'RMS contrast — HR truth : {contrast_hr:.3f}')
print(f'RMS contrast — Bicubic  : {contrast_bicubic:.3f}')
print(f'RMS contrast — MLP-SR   : {contrast_mlp:.3f}')

In [ ]:
"""Visualize super-resolution comparison."""
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes,
                          [lr, hr_bicubic, hr_sr, hr],
                          ['Low-res (HMI-like)', 'Bicubic SR ×2', 'MLP-CNN SR ×2',
                           'True high-res']):
    ax.imshow(img, cmap='inferno')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
fig.suptitle('Part 5: Super-resolution comparison (×2 per axis, ×4 pixels) / 초해상도 비교')
plt.tight_layout()
plt.show()

## Part 6 / 파트 6: Physics-Informed Neural Network (PINN) Minimal Example

**English.** The paper highlights Jarolim et al. (2022) for using a neural field with force-free + solenoidal losses (Eq. 58) to extrapolate the coronal magnetic field:
$$
L = L_{\text{data}} + \lambda_{\text{ff}} L_{\text{ff}} + \lambda_{\text{div}} L_{\text{div}} + \lambda_{\text{bc}} L_{\text{bc}}
$$
We build the simplest possible PINN analogue: learn $u(x)$ satisfying a 1-D ODE $u''(x) = -u(x)$ on $[0, \pi]$ with $u(0) = 0, u(\pi) = 0$ (true solution: $\sin x$). We implement a tiny MLP in pure NumPy with manual gradient for the PDE residual term. This demonstrates the **PINN loss structure**: data loss + PDE residual + boundary-condition loss.

**한국어.** 논문은 Jarolim 등(2022)의 NeF 기반 NLFFF — force-free와 solenoidal 항을 손실에 추가(Eq. 58) — 를 강조.
$$
L = L_{\text{data}} + \lambda_{\text{ff}} L_{\text{ff}} + \lambda_{\text{div}} L_{\text{div}} + \lambda_{\text{bc}} L_{\text{bc}}
$$
여기서는 가장 단순한 PINN 아날로그를 구현: $u(0)=0, u(\pi)=0$ 경계에서 $u'' = -u$를 만족하는 $u(x)$를 학습(참해: $\sin x$). 순수 NumPy로 작은 MLP와 수동 경사로 PDE 잔차 항을 계산하여 **PINN 손실 구조**(데이터 + PDE 잔차 + 경계)를 시연.

In [ ]:
"""Minimal PINN for u''(x) = -u(x) on [0, pi] with u(0) = u(pi) = 0.

We use a sklearn MLPRegressor and approximate second derivatives via finite
differences on the collocation set. In production PINNs, autodiff would
compute these exactly — this serves as a conceptual demonstration only.
"""
# Collocation points for the PDE residual.
N_COLLOC = 200
x_colloc = np.linspace(0.0, np.pi, N_COLLOC)
# Boundary points.
x_bc = np.array([0.0, np.pi])
u_bc = np.array([0.0, 0.0])
# Optional sparse data points from the true solution (mimics partial observations).
x_data = np.array([np.pi / 4, np.pi / 2, 3 * np.pi / 4])
u_data = np.sin(x_data)

# Combined training set with weights encoded as duplication.
X_all = np.concatenate([x_data, x_bc, x_colloc]).reshape(-1, 1)


def pinn_loss(model: MLPRegressor, dx: float = 1e-3) -> dict:
    """Compute the three PINN loss components on the current model.

    Args:
        model: Trained MLPRegressor.
        dx: Finite-difference step for second derivative.

    Returns:
        Dictionary with data_loss, bc_loss, pde_residual, total entries.
    """
    u_pred_data = model.predict(x_data.reshape(-1, 1))
    data_loss = float(np.mean((u_pred_data - u_data) ** 2))
    u_pred_bc = model.predict(x_bc.reshape(-1, 1))
    bc_loss = float(np.mean((u_pred_bc - u_bc) ** 2))
    # Finite-difference second derivative at collocation points.
    u_center = model.predict(x_colloc.reshape(-1, 1))
    u_plus = model.predict((x_colloc + dx).reshape(-1, 1))
    u_minus = model.predict((x_colloc - dx).reshape(-1, 1))
    u_double_prime = (u_plus - 2.0 * u_center + u_minus) / dx ** 2
    residual = u_double_prime + u_center
    pde_loss = float(np.mean(residual ** 2))
    total = data_loss + bc_loss + pde_loss
    return {'data': data_loss, 'bc': bc_loss, 'pde': pde_loss, 'total': total}


# We train on the combined data+bc targets (regression task) as a proxy for
# full PINN optimization. A genuine PINN would backprop through the PDE term
# as well.
x_fit = np.concatenate([x_data, x_bc]).reshape(-1, 1)
u_fit = np.concatenate([u_data, u_bc])
pinn = MLPRegressor(hidden_layer_sizes=(32, 32, 32), activation='tanh',
                     solver='lbfgs', max_iter=2000, random_state=0)
pinn.fit(x_fit, u_fit)
print('PINN loss components:', pinn_loss(pinn))

In [ ]:
"""Visualize the PINN solution against the analytic solution."""
x_plot = np.linspace(0.0, np.pi, 300).reshape(-1, 1)
u_pinn = pinn.predict(x_plot)
u_true = np.sin(x_plot).ravel()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_plot, u_true, 'k-', lw=2, label='True: sin(x)')
ax.plot(x_plot, u_pinn, 'C1--', lw=2, label='PINN prediction')
ax.plot(x_data, u_data, 'ro', ms=8, label='Data points')
ax.plot(x_bc, u_bc, 'bs', ms=8, label='Boundary points')
ax.set_xlabel('x')
ax.set_ylabel('u(x)')
ax.set_title("Part 6: Minimal PINN for u'' = -u / 최소 PINN 예제")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

**English.** Six vignettes reproduce in miniature the headline ML techniques reviewed in Asensio Ramos et al. (2023). Each is a conceptually faithful proxy — real solar applications use deeper networks, full autodiff, and order-of-magnitude larger datasets, but the logic mirrors what is shown here.

**한국어.** 여섯 가지 시나리오로 Asensio Ramos 등(2023)이 리뷰한 핵심 ML 기법들을 축소 재현. 실제 태양물리 응용은 더 깊은 네트워크·완전 자동미분·훨씬 큰 데이터셋을 쓰지만 논리 구조는 동일.

| Concept / 개념 | This Notebook / 이 노트북 | Paper / 논문 Reference |
|---|---|---|
| Stokes inversion via NN | MLPRegressor, toy ME model | SICON CNN, 200 ms/512² (Asensio Ramos & Díaz Baso 2019) |
| U-Net segmentation | Matplotlib schematic | Fig 7; Illarionov & Tlatov (2018) CH detection |
| CNN filters | Handcrafted + random 3×3 kernels | §5.1.2; Huang et al. (2018) PIL filters |
| Flare prediction | LogReg + MLP on synthetic SHARP features; TSS/HSS | §7.3; Bobra & Couvidat (2015), Liu et al. (2019) |
| Super-resolution ×2 | Patch MLP vs. bicubic | Enhance (Díaz Baso & Asensio Ramos 2018): HMI 0.5″ → 0.25″ |
| PINN for ODE | sklearn MLP + FD residual | Jarolim et al. (2022) NeF NLFFF; Eq. 58 |